# Imports

In [180]:
import pandas as pd
import numpy as np
import sklearn

# Data reading

In [181]:
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

# EDA

## Feature Records

1. Numerical Columns
- LotFrontage
- LotArea
- YearBuilt
- YearRemodAdd
- MasVnrArea
- BsmtFinSF1
- BsmtFinSF2
- BsmtUnfSF
- TotalBsmtSF
- 1stFlrSF
- 2ndFlrSF
- LowQualFinSF
- GrLivArea
- BsmtFullBath
- BsmtHalfBath
- FullBath
- HalfBath
- Bedroom
- Kitchen
- TotRmsAbvGrd
- Fireplaces
- GarageYrBlt
- GarageArea
- GarageCars
- WoodDeckSF
- OpenPorchSF
- EnclosedPorch
- 3SsnPorch
- ScreenPorch
- PoolArea
- MiscVal
- MoSold
- YrSold

2. Nominal Columns
- MSZoning
- LotConfig
- Neighborhood
- Condition1
- Condition2
- RoofStyle
- RoofMatl
- Exterior1st
- Exterior2nd
- MasVnrType
- Foundation
- Heating
- MiscFeature

3. Ordinal Columns
- MSSubClass
- LotShape
- Street
- Alley
- LandContour
- Utilities
- LandSlope
- Bldgtype
- HouseStyle
- OverallQual
- OverallCond
- ExterQual
- ExterCond
- BsmtQual
- BsmtCond
- BsmtExposure
- BsmtFinType1
- BsmtFinType2
- HeatingQC
- CentralAir
- Electrical
- KitchenQual
- Functional
- FireplaceQu
- GarageType
- GarageFinish
- GarageQual
- GarageCond
- PavedDrive
- PoolQC
- Fence
- SaleType
- SaleCondition

In [182]:
train_df = train_df.drop(columns="Id")
test_df = test_df.drop(columns="Id")

In [183]:
numeric_columns = {
    "LotFrontage",
    "LotArea",
    "YearBuilt",
    "YearRemodAdd",
    "MasVnrArea",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "LowQualFinSF",
    "GrLivArea",
    "BsmtFullBath",
    "BsmtHalfBath",
    "FullBath",
    "HalfBath",
    "Bedroom",
    "Kitchen",
    "TotRmsAbvGrd",
    "Fireplaces",
    "GarageYrBlt",
    "GarageArea",
    "GarageCars",
    "WoodDeckSF",
    "OpenPorchSF",
    "EnclosedPorch",
    "3SsnPorch",
    "ScreenPorch",
    "PoolArea",
    "MiscVal",
    "MoSold",
    "YrSold",
    "MSSubClass",
    "OverallQual",
    "OverallCond"
}

In [184]:
nominal_columns = {
    "MSZoning",
    "LotConfig",
    "Neighborhood",
    "Condition1",
    "Condition2",
    "RoofStyle",
    "RoofMatl",
    "Exterior1st",
    "Exterior2nd",
    "MasVnrType",
    "Foundation",
    "Heating",
    "MiscFeature",
    "LandContour",
    "SaleType"
}

In [185]:
ordinal_columns = {
    "LotShape": ["IR3", "IR2", "IR1", "Reg"],
    "Street": ["Grvl", "Pave"],
    "Alley": ["NA", "Grvl", "Pave"],
    "Utilities": ["ELO", "NoSeWa", "NoSewr", "AllPub"],
    "LandSlope": ["Gtl", "Mod", "Sev"],
    "Bldgtype": ["1Fam", "2FmCon", "Duplx", "TwnhsE", "TwnhsI"],
    "HouseStyle": ["1Story", "1.5Unf", "1.5Fin",
                   "2Story", "2.5Unf", "2.5Fin", "SFoyer", "SLvl"],
    "ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "ExterCond": ["Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtQual": ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtCond": ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtExposure": ["NA", "No", "Mn", "Av", "Gd"],
    "BsmtFinType1": ["NA", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "BsmtFinType2": ["NA", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "HeatingQC": ["Po", "Fa", "TA", "Gd", "Ex"],
    "CentralAir": ["N", "Y"],
    "Electrical": ["SBrkr", "FuseP", "FuseF", "FuseA", "Mix"],
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "Functional": ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    "FireplaceQu": ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "GarageType": ["NA", "Detchd", "CarPort", "BuiltIn", "Basment", "Attchd", "2Types"],
    "GarageFinish": ["NA", "Unf", "RFn", "Fin"],
    "GarageQual": ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "GarageCond": ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "PavedDrive": ["N", "P", "Y"],
    "PoolQC": ["NA", "Po", "Fa", "TA", "Gd", "Ex"],
    "Fence": ["NA", "MnWw", "GdWo", "MnPrv", "GdPrv"],
    "SaleCondition": ["Abnorml", "Partial", "Family", "Alloca", "AdjLand", "Normal"],
}

## Missing Data

In [186]:
def clear_sparse(df):
    l = df.columns[df.isna().any()].tolist()
    k = []
    for i in l:
        if 100 * df[i].isna().sum() / len(df[i]) >= 30:
            k.append(i)
    df = df.drop(columns=k)

    missing_cols = list(set(l).difference(set(k)))

    return df, missing_cols

In [187]:
def feature_split(cols):
    numeric = set(cols).intersection(numeric_columns)
    nominal = set(cols).intersection(nominal_columns)
    ordinal = set(cols).intersection(ordinal_columns.keys())

    return numeric, nominal, ordinal

In [188]:
from sklearn.impute import SimpleImputer
def simple_impute(df, cols, type, missing_val=np.nan, fill_val = None):
    imputer = SimpleImputer(strategy=type, missing_values=missing_val,
                            fill_value=fill_val)
    df[cols] = imputer.fit_transform(df[cols])

    return df, imputer

In [189]:
train_df, train_miss = clear_sparse(train_df)
test_df, test_miss = clear_sparse(test_df)

In [190]:
train_numeric_miss, train_nominal_miss, train_ordinal_miss = feature_split(train_miss)
test_numeric_miss, test_nominal_miss, test_ordinal_miss = feature_split(test_miss)

In [191]:
train_df, train_numeric_imputer = simple_impute(train_df,
                                                list(train_numeric_miss),
                                                type="median")
test_df, test_numeric_imputer = simple_impute(test_df,
                                              list(test_numeric_miss),
                                              "median")

In [192]:
train_df, train_ordinal_imputer = simple_impute(train_df,
                                                list(train_ordinal_miss),
                                                "most_frequent")
test_df, test_ordinal_imputer = simple_impute(test_df,
                                              list(test_ordinal_miss),
                                              "most_frequent")

## Encode

In [193]:
import pandas as pd
from sklearn.feature_extraction import FeatureHasher

def nominal_encode(df, cols):
    cols = list(set(df.columns).intersection(set(cols)))
    dict_data = df[cols].astype(str).to_dict(orient='records')
    hasher = FeatureHasher(n_features=len(cols), input_type='dict')
    df[cols] = hasher.transform(dict_data).toarray()

    return df, hasher

In [194]:
from sklearn.preprocessing import OrdinalEncoder
def ordinal_encode(df, ord_dict):
    for col, cats in ord_dict.items():
        if col in df.columns:
            ord_enc = OrdinalEncoder(categories=[cats])
            df[col] = ord_enc.fit_transform(df[[col]])
    return df, ord_enc

In [195]:
train_df, train_ord_enc = ordinal_encode(train_df, ordinal_columns)
test_df, test_ord_enc = ordinal_encode(test_df, ordinal_columns)

In [196]:
train_df, train_nom_enc = nominal_encode(train_df, list(nominal_columns))
test_df, test_nom_enc = nominal_encode(test_df, list(nominal_columns))

In [197]:
train_df

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,60,0.0,65.0,8450,1.0,3.0,0.0,3.0,2.0,0.0,1.0,0.0,0.0,1Fam,3.0,7,5,2003,2003,0.0,0.0,2.0,-3.0,196.0,3.0,2.0,0.0,4.0,3.0,1.0,6.0,706,1.0,0,150,856,0.0,4.0,1.0,0.0,856,854,0,1710,1,0,2,1,3,1,3.0,8,7.0,0,5.0,2003.0,2.0,2,548,3.0,3.0,2.0,0,61,0,0,0,0,0,2,2008,-1.0,5.0,208500
1,20,0.0,80.0,9600,1.0,3.0,0.0,3.0,2.0,0.0,-1.0,-1.0,2.0,1Fam,0.0,6,8,1976,1976,-1.0,-1.0,1.0,-2.0,0.0,2.0,2.0,0.0,4.0,3.0,4.0,5.0,978,1.0,0,284,1262,-2.0,4.0,1.0,0.0,1262,0,0,1262,0,1,2,0,3,1,2.0,6,7.0,1,5.0,1976.0,2.0,2,460,3.0,3.0,2.0,298,0,0,0,0,0,0,5,2007,0.0,5.0,181500
2,60,0.0,68.0,11250,1.0,2.0,0.0,3.0,2.0,0.0,1.0,0.0,0.0,1Fam,3.0,7,5,2001,2002,0.0,0.0,2.0,-3.0,162.0,3.0,2.0,0.0,4.0,3.0,2.0,6.0,486,1.0,0,434,920,0.0,4.0,1.0,0.0,920,866,0,1786,1,0,2,1,3,1,3.0,6,7.0,1,5.0,2001.0,2.0,2,608,3.0,3.0,2.0,0,42,0,0,0,0,0,9,2008,-1.0,5.0,223500
3,70,0.0,60.0,9550,1.0,2.0,1.0,3.0,3.0,0.0,0.0,-1.0,0.0,1Fam,3.0,7,5,1915,1970,0.0,0.0,1.0,-2.0,0.0,2.0,2.0,-1.0,3.0,4.0,1.0,5.0,216,1.0,0,540,756,0.0,3.0,1.0,0.0,961,756,0,1717,1,0,1,0,3,1,3.0,7,7.0,1,1.0,1998.0,1.0,3,642,3.0,3.0,2.0,0,35,272,0,0,0,0,2,2006,0.0,0.0,140000
4,60,0.0,84.0,14260,1.0,2.0,0.0,3.0,2.0,0.0,1.0,-1.0,0.0,1Fam,3.0,8,5,2000,2000,0.0,0.0,1.0,-2.0,350.0,3.0,2.0,0.0,4.0,3.0,3.0,6.0,655,1.0,0,490,1145,-1.0,4.0,1.0,0.0,1145,1053,0,2198,1,0,2,1,4,1,3.0,9,7.0,1,5.0,2000.0,2.0,3,836,3.0,3.0,2.0,192,84,0,0,0,0,0,12,2008,-1.0,5.0,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,60,0.0,62.0,7917,1.0,3.0,0.0,3.0,2.0,0.0,1.0,0.0,0.0,1Fam,3.0,6,5,1999,2000,0.0,1.0,2.0,-2.0,0.0,2.0,2.0,0.0,4.0,3.0,1.0,1.0,0,1.0,0,953,953,0.0,4.0,1.0,0.0,953,694,0,1647,0,0,2,1,3,1,2.0,7,7.0,1,5.0,1999.0,2.0,2,460,3.0,3.0,2.0,0,40,0,0,0,0,0,8,2007,-1.0,5.0,175000
1456,20,0.0,85.0,13175,1.0,3.0,1.0,3.0,1.0,0.0,0.0,0.0,1.0,1Fam,0.0,6,6,1978,1988,1.0,0.0,1.0,-2.0,119.0,2.0,2.0,0.0,4.0,3.0,1.0,5.0,790,3.0,163,589,1542,0.0,2.0,1.0,0.0,2073,0,0,2073,1,0,2,0,3,1,2.0,7,6.0,2,5.0,1978.0,1.0,2,500,3.0,3.0,2.0,349,0,0,0,0,0,0,2,2010,0.0,5.0,210000
1457,70,0.0,66.0,9042,1.0,3.0,0.0,3.0,3.0,0.0,0.0,0.0,0.0,1Fam,3.0,7,9,1941,2006,0.0,0.0,1.0,-2.0,0.0,4.0,3.0,0.0,3.0,4.0,1.0,6.0,275,1.0,0,877,1152,1.0,4.0,1.0,0.0,1188,1152,0,2340,0,0,2,0,4,1,3.0,9,7.0,2,5.0,1941.0,2.0,1,252,3.0,3.0,2.0,0,60,0,0,0,0,2500,5,2010,0.0,5.0,266500
1458,20,0.0,68.0,9717,1.0,3.0,-1.0,3.0,2.0,0.0,1.0,0.0,2.0,1Fam,0.0,5,6,1950,1996,0.0,0.0,1.0,-1.0,0.0,2.0,2.0,0.0,3.0,3.0,2.0,6.0,49,3.0,1029,0,1078,-1.0,3.0,1.0,3.0,1078,0,0,1078,1,0,1,0,2,1,3.0,5,7.0,0,5.0,1950.0,1.0,1,240,3.0,3.0,2.0,366,0,112,0,0,0,0,4,2010,0.0,5.0,142125


# Feature Visualization

# Feature Transformation

In [205]:
target = "SalePrice"
train_X, train_y = train_df.drop(columns=target), train_df[target]

In [204]:
from sklearn.preprocessing import RobustScaler

def numeric_scale(df):
    cols = list(set(df.columns).intersection(numeric_columns))
    scaler = RobustScaler()
    df[cols] = scaler.fit_transform(df[cols])

    return df, scaler

In [206]:
train_X, train_numeric_scaler = numeric_scale(train_X)
test_df, test_numeric_scaler = numeric_scale(test_df)

# Feature Selection

# Train